In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import tqdm
import torch
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as tfs

from CNN import ConvNet, ConvNetTrainer

In [ ]:
# CIFAR-10

BATCH_SIZE = 20  
NUM_WORKERS = 4  # threads 2~8

# data augmentation
transform_train = tfs.Compose([
    tfs.RandomCrop(32, padding=4),
    tfs.RandomHorizontalFlip(),
    tfs.ColorJitter(brightness=0.2, contrast=0.2),
    tfs.ToTensor(),
    tfs.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = tfs.Compose([
    tfs.ToTensor(),  
    tfs.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_set = datasets.CIFAR10(root='../../Data/CIFAR10', train=True, download=True, transform=transform_train)
test_set = datasets.CIFAR10(root='../../Data/CIFAR10', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(
    dataset=train_set,        
    batch_size=BATCH_SIZE,    
    shuffle=True,             
    num_workers=NUM_WORKERS,  
    pin_memory=True,          #  CPU to GPU memory
    drop_last=False           
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_set,         
    batch_size=BATCH_SIZE,    
    shuffle=False,            
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

In [ ]:
model_save_root = "../../Models/CNN"
os.makedirs(model_save_root,exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

# CNN

#### Train

In [ ]:
net=ConvNet(num_class=10)

lr=1e-4

optimizer = optim.Adam(net.parameters(), lr=lr)

trainer=ConvNetTrainer(model=net,
                       optimizer=optimizer,
                       dtype=float,
                       device=device)

trainer.train(num_epochs=10,
              train_dataloader=train_loader,
              log_interval=50)

torch.save(trainer.model.state_dict(),os.path.join(model_save_root,"CNN.pth"))

#### Evaluation

In [ ]:
net=ConvNet(num_class=10).to(device)
net.load_state_dict(torch.load(os.path.join(model_save_root,"CNN.pth"), map_location=device))
net.eval()

correct = 0        
total = 0        

with torch.no_grad():   
    for imgs, labels in tqdm.tqdm(test_loader):
        imgs = imgs.to(device)
        labels = labels.to(device)

        logits = net(imgs)           

        preds = logits.argmax(dim=1)  # (B,)
        correct += (preds == labels).sum().item()   
        total += labels.shape[0]                     

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")   